# 06 · (Bonus) Streamlit Demo on Colab
Launches `app.py` and gives a public link through a Cloudflare tunnel.
The **Results Explorer** works from the saved results; **Analyze a New Call** needs the GPU runtime and both secrets.

In [1]:
# ---- Setup: open the project folder (run this cell first) ----
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")                       # results are saved to Drive, so they survive restarts
    PROJECT_DIR = "/content/drive/MyDrive/call-center-ai"  # <- change if your folder is elsewhere
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Working in:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/call-center-ai


In [ ]:
!pip install -q streamlit openai-whisper pyannote.audio jiwer groq soundfile torchvision torchaudio
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import os; os.kill(os.getpid(), 9)   # restart after installing (keeps torch / torchvision in sync); then run Setup again

In [2]:
!nvidia-smi --query-gpu=name --format=csv
import torch; print("torch", torch.__version__, "| CUDA build:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())

/bin/bash: line 1: nvidia-smi: command not found
torch 2.11.0+cpu | CUDA build: None | GPU available: False


In [2]:
# Direct test of the full pipeline on one call (shows any error clearly)
import time, torch
from src import config
from src.llm import GroqLLM
from src.transcriber import WhisperTranscriber
from src.diarizer import SpeakerDiarizer
from src.role_labeler import RoleLabeler
from src.summarizer import CallSummarizer
from src.sentiment import SentimentAnalyzer
from src.pipeline import CallCenterPipeline

print("GPU:", torch.cuda.is_available())
t0 = time.time()
llm = GroqLLM()
pipe = CallCenterPipeline(WhisperTranscriber(), SpeakerDiarizer(config.get_secret("HF_TOKEN")),
                          RoleLabeler(llm), CallSummarizer(llm), SentimentAnalyzer(llm))
print(f"Models loaded in {time.time() - t0:.0f}s")
res = pipe.process_call("test_call", config.AUDIO_DIR / "call_011.wav")
print(f"Done in {time.time() - t0:.0f}s")
print("Roles:", res.speaker_roles)
print("Summary:", res.summary["summary"])
print("Sentiment:", res.sentiment["overall_sentiment"], "-", res.sentiment["justification"])

GPU: False
Groq model: openai/gpt-oss-120b (fallbacks: ['openai/gpt-oss-20b', 'qwen/qwen3.8-27b'])
Models loaded in 39s


KeyboardInterrupt: 

In [2]:
# Pass the Colab secrets to the app as environment variables (never printed or saved)
from src import config
for k in ["GROQ_API_KEY", "HF_TOKEN"]:
    v = config.get_secret(k)
    if v: os.environ[k] = v
!PYTHONUNBUFFERED=1 nohup streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &
print('App log: /content/streamlit.log (Files panel) - shows progress while a call is analysed')

App log: /content/streamlit.log (Files panel) - shows progress while a call is analysed


In [ ]:
# Open the https://....trycloudflare.com link printed below. Stop this cell to shut the demo down.
!./cloudflared tunnel --url http://localhost:8501

2026-11-02T15:03:28Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-02T15:03:28Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-02T15:03:33Z INF +--------------------------------------------------------------------------------------------+
2026-11-02T15:03:33Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-02T15:03:33Z INF |  https://plain-enable-registry-destination.trycloudfla